[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/01-python/12_exceptions.ipynb)

*Python from Scratch · Notebook 12 of 12*

# Errors, Exceptions & Files

Handle things that go wrong with try / except / else / finally, raise your own errors, and read and write files safely.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** An exception is a fire alarm: normal work stops, and control jumps to whoever is ready to handle it.

**What interviewers really test here:** Catching specific errors, else/finally, context managers and reading files safely.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Errors are normal *(Basics)*
- [ ] 2. try and except *(Basics)*
- [ ] 3. else, finally and several excepts *(Core)*
- [ ] 4. Raising your own errors *(Core)*
- [ ] 5. Reading and writing files *(Core)*
- [ ] 6. Context managers and structured data files *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definitions

- An **error** is a problem in a program. **Syntax errors** are detected before the program runs; everything else happens at run time.
- An **exception** is an object that represents a run-time error. When one is **raised**, normal execution stops and Python looks for a handler up the call stack; if none is found, the program terminates and prints a **traceback**.
- **Exception handling** is catching an exception with `try`/`except` and responding to it so the program can continue or fail gracefully.

### The exception hierarchy

```text
BaseException
 ├── SystemExit, KeyboardInterrupt, GeneratorExit
 └── Exception
      ├── ArithmeticError → ZeroDivisionError, OverflowError
      ├── LookupError → IndexError, KeyError
      ├── ValueError → UnicodeError …
      ├── TypeError
      ├── NameError → UnboundLocalError
      ├── AttributeError
      ├── OSError → FileNotFoundError, PermissionError …
      ├── ImportError → ModuleNotFoundError
      ├── RuntimeError → RecursionError, NotImplementedError
      └── StopIteration …
```



An `except` clause catches the named class and all its subclasses: `except LookupError` catches both IndexError and KeyError. Your own exceptions should inherit from `Exception`, not BaseException.

### try statement semantics

| Clause | Runs when |
| --- | --- |
| `try` | always first |
| `except E [as e]` | an exception of type E (or a subclass) was raised in try; the first matching clause wins |
| `else` | try finished **without** an exception |
| `finally` | always, last — even after return, break or an unhandled exception |




- Put the most specific exception classes first.
- Keep the try block small — only the statement that can fail.
- `raise` with no argument inside an except re-raises the current exception.
- `raise NewError(...) from e` chains exceptions so the original cause is kept.
- Python 3.11+ adds `ExceptionGroup` and `except*` for multiple simultaneous errors.

### Raising and custom exceptions

```python
raise ValueError("price must be positive")

class AppError(Exception):
    """Base class for this application's errors."""

class InsufficientFunds(AppError):
    def __init__(self, needed):
        super().__init__(f"need {needed} more")
        self.needed = needed
```



`assert condition, "message"` raises AssertionError when the condition is false. Use it for internal sanity checks, not for validating user input (asserts can be disabled with `python -O`).

### LBYL vs EAFP

|   | LBYL (look before you leap) | EAFP (easier to ask forgiveness) |
| --- | --- | --- |
| Style | `if key in d: x = d[key]` | `try: x = d[key] except KeyError: …` |
| Pros | explicit; no exception overhead when failures are common | no duplicated checks; avoids race conditions (file deleted between check and use) |
| Typical in | C, Java | Python |

### File handling

| Task | Code |
| --- | --- |
| Open safely | `with open(path, mode, encoding="utf-8") as f:` |
| Read all | `f.read()` |
| Read lines lazily | `for line in f:` |
| List of lines | `f.readlines()` or `f.read().splitlines()` |
| Write | `f.write(text)`, `f.writelines(lines)`, `print(x, file=f)` |
| Modes | r, w, a, x, + (read/write), b (binary), t (text, default) |
| Paths | `pathlib.Path("data") / "file.txt"`, `.exists()`, `.read_text()` |
| CSV / JSON | `csv.reader`, `csv.DictReader`; `json.load`, `json.dump` |



A **context manager** implements `__enter__` and `__exit__`; `with` guarantees `__exit__` runs, so resources are released. Without `with`, you must call `f.close()` yourself (in a finally block).

---
# Part 2: Lessons, step by step

## Lesson 1: Errors are normal

*Level: Basics*

Programs meet bad situations: a user types "abc" where a number is expected, a file is missing, a list is shorter than you thought. Python reports these as **exceptions**. If nobody handles one, the program stops with a **traceback**:



```python
Traceback (most recent call last):
  File "app.py", line 3, in <module>
    age = int(input("Age: "))
ValueError: invalid literal for int() with base 10: 'abc'
```



Read a traceback from the **bottom**: the last line names the error type and message; the lines above show where it happened.



| Error | Typical cause |
| --- | --- |
| `SyntaxError` | code isn't valid Python (missing colon, bracket) |
| `NameError` | using a name that doesn't exist (typo) |
| `TypeError` | wrong type, e.g. `"5" + 3` |
| `ValueError` | right type, bad value, e.g. `int("abc")` |
| `IndexError` / `KeyError` | list index / dict key doesn't exist |
| `ZeroDivisionError` | dividing by zero |
| `FileNotFoundError` | opening a file that isn't there |
| `AttributeError` | object has no such attribute/method |

## Lesson 2: try and except

*Level: Basics*

Put risky code inside `try`. If an exception happens, Python jumps to a matching `except` block instead of crashing:



```python
try:
    age = int(text)
except ValueError:
    age = 0
    print("Not a number, using 0")
```



Catch **specific** exceptions. A bare `except:` also catches typos and Ctrl+C, hiding real bugs.



Two styles: **LBYL** ("look before you leap": check with if first) and **EAFP** ("easier to ask forgiveness than permission": just try it and handle the error). EAFP is the usual Python style.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def try_demo():
    results = []
    for text in ["42", "abc", "7"]:
        try:
            n = int(text)
            results.append(n)
        except ValueError:
            print("skipping", repr(text))
    print(results)


try_demo()

skipping 'abc'
[42, 7]


**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q3](#q3), [Q4](#q4), [Q6](#q6). Try one now, or carry on and do them in Part 3.

## Lesson 3: else, finally and several excepts

*Level: Core*

```python
try:
    value = int(items[i])
except IndexError:
    print("no such position")
except ValueError as e:          # 'as e' gives you the exception object
    print("bad number:", e)
else:
    print("got", value)           # only when NO exception happened
finally:
    print("this always runs")     # clean-up: close files, release locks
```



Order: try → (one matching except, or else) → finally. `finally` runs even if the try block returns or an unhandled error is on its way out.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def full_try_demo():
    log = []
    for value in [4, 0]:
        try:
            result = 100 / value
        except ZeroDivisionError as e:
            log.append(f"error: {e}")
        else:
            log.append(f"result {result}")
        finally:
            log.append("finished")
    print(log)


full_try_demo()

['result 25.0', 'finished', 'error: division by zero', 'finished']


**✏️ Your turn:** you now have what you need for [Q16](#q16), [Q5](#q5), [Q14](#q14), [Q11](#q11), [Q15](#q15). Try one now, or carry on and do them in Part 3.

## Lesson 4: Raising your own errors

*Level: Core*

When your function gets input it can't handle, **raise** an exception instead of returning a strange value:



```python
def set_age(age):
    if age < 0:
        raise ValueError("age cannot be negative")
```



You can define your own exception types by subclassing `Exception`:



```python
class InsufficientFunds(Exception):
    pass

raise InsufficientFunds("balance too low")
```



This separates **detecting** a problem (deep inside your logic) from **deciding what to do** about it (at the top, near the user).

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def raise_demo():
    class InsufficientFunds(Exception):
        pass

    def withdraw(balance, amount):
        if amount > balance:
            raise InsufficientFunds(f"need {amount - balance} more")
        return balance - amount

    try:
        left = withdraw(100, 30)
        left = withdraw(left, 500)
    except InsufficientFunds as e:
        print("Sorry:", e, "| balance still", left)


raise_demo()

Sorry: need 430 more | balance still 70


<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. InsufficientFunds() returns None
2. withdraw() returns 70
3. InsufficientFunds raised: need 430 more
4. withdraw() returns None
5. InsufficientFunds raised: need 430 more
6. raise\_demo() returns None

</details>


**✏️ Your turn:** you now have what you need for [Q12](#q12), [Q13](#q13). Try one now, or carry on and do them in Part 3.

## Lesson 5: Reading and writing files

*Level: Core*

`open(path, mode)` returns a file object. Always use it in a `with` block — the file is closed automatically, even if an error happens.



| Mode | Meaning |
| --- | --- |
| `"r"` | read text (default); error if missing |
| `"w"` | write text; creates or **empties** the file |
| `"a"` | append to the end |
| `"x"` | create; error if it exists |
| `"b"` (add to others) | binary, e.g. `"rb"` for images |



```python
with open("notes.txt", encoding="utf-8") as f:
    for line in f:                 # one line at a time — memory friendly
        print(line.rstrip("\n"))

with open("out.txt", "w", encoding="utf-8") as f:
    f.write("first line\n")
```



Other reads: `f.read()` (everything as one string), `f.readline()`, `f.readlines()` (list of lines). The practice questions use `io.StringIO`, an in-memory file that behaves exactly like an open text file, so they can run anywhere.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [5]:
def file_demo():
    import io
    fake_file = io.StringIO("apples 3\npears 5\nplums 2\n")
    total = 0
    with fake_file as f:
        for line in f:
            name, qty = line.split()
            total += int(qty)
    print("total fruit:", total)


file_demo()

total fruit: 10


**✏️ Your turn:** you now have what you need for [Q7](#q7), [Q8](#q8), [Q10](#q10), [Q9](#q9). Try one now, or carry on and do them in Part 3.

## Lesson 6: Context managers and structured data files

*Level: Advanced*

A **context manager** is any object usable with `with`: it runs set-up code on entry (`__enter__`) and clean-up on exit (`__exit__`), even after an error. Files, locks and database connections are context managers. The easiest way to write your own:



```python
from contextlib import contextmanager
@contextmanager
def timer():
    start = time.time()
    try:
        yield                      # the with-block runs here
    finally:
        print(time.time() - start)
```



For structured files use the standard library instead of splitting by hand: `csv` for comma-separated data, `json` for JSON (`json.load`, `json.dump`, catching `json.JSONDecodeError`), and `pathlib.Path` for working with paths.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [6]:
def contextmanager_demo():
    from contextlib import contextmanager
    log = []

    @contextmanager
    def session(name):
        log.append("open " + name)
        try:
            yield name.upper()
        finally:
            log.append("close " + name)

    try:
        with session("db") as s:
            log.append("using " + s)
            raise RuntimeError("lost connection")
    except RuntimeError as e:
        log.append("handled: " + str(e))
    print(log)


contextmanager_demo()

['open db', 'using DB', 'close db', 'handled: lost connection']


<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. session() returns 'DB'
2. RuntimeError raised: lost connection
3. RuntimeError raised: lost connection
4. session() returns None
5. contextmanager\_demo() returns None

</details>


**✏️ Your turn:** you now have what you need for [Q20](#q20), [Q17](#q17), [Q19](#q19), [Q18](#q18). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Safe Division

🟢 Warm-up · *ZeroDivisionError*

Return a / b, or None if b is zero.

**Examples**

- `a = 10, b = 2` → `5.0`
- `a = 1, b = 0` → `None`
- `a = -9, b = 3` → `-3.0`

In [ ]:
def safe_divide(a, b):
    """
    Return a / b, or None if b is zero.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(safe_divide, [
    ((10, 2), 5.0),
    ((1, 0), None),
    ((-9, 3), -3.0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Look before you leap (LBYL): check b before dividing.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Easier to ask forgiveness (EAFP): try the division and catch `ZeroDivisionError`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Look before you leap (LBYL): check b before dividing.

```python
def safe_divide(a, b):
    if b == 0:
        return None
    return a / b
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Easier to ask forgiveness (EAFP): try the division and catch `ZeroDivisionError`.

```python
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return None
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Look before you leap (LBYL): check b before dividing. That costs O(1) time and O(1) space."

"I can do better: Easier to ask forgiveness (EAFP): try the division and catch `ZeroDivisionError`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/exceptions/safe-division)

<a id="q2"></a>
### Q2. Parse a Number or Use a Default

🟢 Warm-up · *ValueError*

Convert `text` to an int; if it isn't a valid integer return `default`.

**Examples**

- `text = '42', default = 0` → `42`
- `text = '4x', default = -1` → `-1`
- `text = ' -7 ', default = 0` → `-7`

In [ ]:
def to_int(text, default):
    """
    Convert text to an int; if it isn't a valid integer return default.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(to_int, [
    (('42', 0), 42),
    (('4x', -1), -1),
    ((' -7 ', 0), -7),
    (('', 5), 5),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check the characters yourself (strip, optional sign, all digits).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Just call `int()` and catch `ValueError` — int already knows every rule.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check the characters yourself (strip, optional sign, all digits).

```python
def to_int(text, default):
    t = text.strip()
    if t.startswith("-") or t.startswith("+"):
        digits = t[1:]
    else:
        digits = t
    if digits == "" or not digits.isdigit():
        return default
    return int(t)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Just call `int()` and catch `ValueError` — int already knows every rule.

```python
def to_int(text, default):
    try:
        return int(text)
    except ValueError:
        return default
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check the characters yourself (strip, optional sign, all digits). That costs O(n) time and O(1) space."

"I can do better: Just call `int()` and catch `ValueError` — int already knows every rule. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/exceptions/parse-a-number-or-use-a-default)

<a id="q3"></a>
### Q3. Safe List Access

🟢 Warm-up · *IndexError*

Return `items[i]`, or `default` if i is out of range.

**Examples**

- `items = [1, 2, 3], i = 1, default = None` → `2`
- `items = [1, 2, 3], i = 9, default = 'none'` → `'none'`
- `items = [1, 2, 3], i = -1, default = 0` → `3`

In [ ]:
def item_or(items, i, default):
    """
    Return items[i], or default if i is out of range.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(item_or, [
    (([1, 2, 3], 1, None), 2),
    (([1, 2, 3], 9, 'none'), 'none'),
    (([1, 2, 3], -1, 0), 3),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check the index range (including negative indices) first.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Try the index and catch `IndexError`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check the index range (including negative indices) first.

```python
def item_or(items, i, default):
    if -len(items) <= i < len(items):
        return items[i]
    return default
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Try the index and catch `IndexError`.

```python
def item_or(items, i, default):
    try:
        return items[i]
    except IndexError:
        return default
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check the index range (including negative indices) first. That costs O(1) time and O(1) space."

"I can do better: Try the index and catch `IndexError`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/exceptions/safe-list-access)

<a id="q4"></a>
### Q4. Safe Dictionary Lookup

🟢 Warm-up · *KeyError*

Return the phone number for `name`, or 'unknown' if the name isn't in the book.

**Examples**

- `book = {'ana': '123', 'ben': '456'}, name = 'ben'` → `'456'`
- `book = {'ana': '123'}, name = 'cy'` → `'unknown'`

In [ ]:
def phone_of(book, name):
    """
    Return the phone number for name, or 'unknown' if the name isn't in the
    book.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(phone_of, [
    (({'ana': '123', 'ben': '456'}, 'ben'), '456'),
    (({'ana': '123'}, 'cy'), 'unknown'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Catch `KeyError` around the lookup.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

For a simple default, `dict.get` is clearer than any exception handling.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Catch `KeyError` around the lookup.

```python
def phone_of(book, name):
    try:
        return book[name]
    except KeyError:
        return "unknown"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** For a simple default, `dict.get` is clearer than any exception handling.

```python
def phone_of(book, name):
    return book.get(name, "unknown")
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Catch `KeyError` around the lookup. That costs O(1) time and O(1) space."

"I can do better: For a simple default, `dict.get` is clearer than any exception handling. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/exceptions/safe-dictionary-lookup)

<a id="q5"></a>
### Q5. Success Message With else

🟢 Warm-up · *try / except / else*

Return 'got N' if text is an integer N, else 'bad input'.

**Examples**

- `text = '12'` → `'got 12'`
- `text = '1.5'` → `'bad input'`

In [ ]:
def describe_input(text):
    """
    Return 'got N' if text is an integer N, else 'bad input'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(describe_input, [
    (('12',), 'got 12'),
    (('1.5',), 'bad input'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Put the success code inside try as well.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep only the risky call in `try`; success code goes in `else`, so its own errors aren't accidentally caught.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Put the success code inside try as well.

```python
def describe_input(text):
    try:
        n = int(text)
        message = "got " + str(n)
        return message
    except ValueError:
        return "bad input"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Keep only the risky call in `try`; success code goes in `else`, so its own errors aren't accidentally caught.

```python
def describe_input(text):
    try:
        n = int(text)
    except ValueError:
        return "bad input"
    else:
        return f"got {n}"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Put the success code inside try as well. That costs O(1) time and O(1) space."

"I can do better: Keep only the risky call in `try`; success code goes in `else`, so its own errors aren't accidentally caught. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/exceptions/success-message-with-else)

<a id="q6"></a>
### Q6. Square Root of a Negative

🟢 Warm-up · *Raising errors*

Return the square root rounded to 3 decimals, or the message 'negative input' — the optimal answer raises and catches ValueError.

**Examples**

- `x = 16` → `4.0`
- `x = 2` → `1.414`
- `x = -4` → `'negative input'`

In [ ]:
def root(x):
    """
    Return the square root rounded to 3 decimals, or the message 'negative
    input' — the optimal answer raises and catches ValueError.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(root, [
    ((16,), 4.0),
    ((2,), 1.414),
    ((-4,), 'negative input'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Return the message from an if branch.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`math.sqrt` already raises ValueError for negatives; catch it.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Return the message from an if branch.

```python
def root(x):
    if x < 0:
        return "negative input"
    return round(x ** 0.5, 3)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `math.sqrt` already raises ValueError for negatives; catch it.

```python
def root(x):
    import math
    try:
        return round(math.sqrt(x), 3)
    except ValueError:
        return "negative input"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Return the message from an if branch. That costs O(1) time and O(1) space."

"I can do better: `math.sqrt` already raises ValueError for negatives; catch it. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/exceptions/square-root-of-a-negative)

<a id="q7"></a>
### Q7. Count Lines in a File

🟢 Warm-up · *Iterating a file*

`text` is the content of a file. Count its lines (simulated with io.StringIO, which behaves like an open file).

**Examples**

- `text = 'a\nb\nc\n'` → `3`
- `text = ''` → `0`
- `text = 'one line'` → `1`

In [ ]:
def count_lines(text):
    """
    text is the content of a file. Count its lines (simulated with io.StringIO,
    which behaves like an open file).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(count_lines, [
    (('a\nb\nc\n',), 3),
    (('',), 0),
    (('one line',), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Read every line into a list with `readlines()` and take its length.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Iterate the file object directly — one line at a time, no list in memory.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Read every line into a list with `readlines()` and take its length.

```python
def count_lines(text):
    import io
    f = io.StringIO(text)
    lines = f.readlines()
    f.close()
    return len(lines)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Iterate the file object directly — one line at a time, no list in memory.

```python
def count_lines(text):
    import io
    with io.StringIO(text) as f:
        return sum(1 for _ in f)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Read every line into a list with `readlines()` and take its length. That costs O(n) time and O(n) space."

"I can do better: Iterate the file object directly — one line at a time, no list in memory. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/exceptions/count-lines-in-a-file)

<a id="q8"></a>
### Q8. Words in a File

🟢 Warm-up · *Reading text*

Return the number of words in the file content.

**Examples**

- `text = 'hello world\nhow are you\n'` → `5`
- `text = ''` → `0`

In [ ]:
def words_in_file(text):
    """
    Return the number of words in the file content.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(words_in_file, [
    (('hello world\nhow are you\n',), 5),
    (('',), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Read line by line and split each line, adding the counts.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Read the whole file and split once.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Read line by line and split each line, adding the counts.

```python
def words_in_file(text):
    import io
    count = 0
    f = io.StringIO(text)
    for line in f:
        count += len(line.split())
    f.close()
    return count
```

**Cost:** O(n) time, O(line) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Read the whole file and split once.

```python
def words_in_file(text):
    import io
    with io.StringIO(text) as f:
        return len(f.read().split())
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Read line by line and split each line, adding the counts. That costs O(n) time and O(line) space."

"I can do better: Read the whole file and split once. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/exceptions/words-in-a-file)

<a id="q9"></a>
### Q9. Write Lines, Read Them Back

🟢 Warm-up · *Writing files*

Write each item on its own line to a (simulated) file, then return the file's full content.

**Examples**

- `items = ['a', 'b']` → `'a\nb\n'`
- `items = []` → `''`

In [ ]:
def write_lines(items):
    """
    Write each item on its own line to a (simulated) file, then return the
    file's full content.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(write_lines, [
    ((['a', 'b'],), 'a\nb\n'),
    (([],), ''),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Call `write` twice per item (text, then newline).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`writelines` with a generator, inside a `with` block.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Call `write` twice per item (text, then newline).

```python
def write_lines(items):
    import io
    f = io.StringIO()
    for item in items:
        f.write(item)
        f.write("\n")
    content = f.getvalue()
    f.close()
    return content
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `writelines` with a generator, inside a `with` block.

```python
def write_lines(items):
    import io
    with io.StringIO() as f:
        f.writelines(f"{item}\n" for item in items)
        return f.getvalue()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Call `write` twice per item (text, then newline). That costs O(n) time and O(n) space."

"I can do better: `writelines` with a generator, inside a `with` block. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/exceptions/write-lines-read-them-back)

<a id="q10"></a>
### Q10. Longest Line

🟢 Warm-up · *max over a file*

Return the longest line of the file without its newline ('' for an empty file).

**Examples**

- `text = 'hi\nhello\nhey\n'` → `'hello'`
- `text = ''` → `''`

In [ ]:
def longest_line(text):
    """
    Return the longest line of the file without its newline ('' for an empty
    file).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(longest_line, [
    (('hi\nhello\nhey\n',), 'hello'),
    (('',), ''),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Track the longest while reading line by line.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`max` over the stripped lines with `key=len` and a default.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Track the longest while reading line by line.

```python
def longest_line(text):
    import io
    best = ""
    f = io.StringIO(text)
    for line in f:
        line = line.rstrip("\n")
        if len(line) > len(best):
            best = line
    f.close()
    return best
```

**Cost:** O(n) time, O(line) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `max` over the stripped lines with `key=len` and a default.

```python
def longest_line(text):
    import io
    with io.StringIO(text) as f:
        return max((line.rstrip("\n") for line in f), key=len, default="")
```

**Cost:** O(n) time, O(line) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Track the longest while reading line by line. That costs O(n) time and O(line) space."

"I can do better: `max` over the stripped lines with `key=len` and a default. That's O(n) time and O(line) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/exceptions/longest-line)

<a id="q11"></a>
### Q11. Sum the Valid Numbers

🟡 Interview standard · *Skipping bad input*

Given strings, return the sum of those that are valid numbers (ints or floats) and how many were skipped: `(total, skipped)`.

**Examples**

- `texts = ['3', 'x', '2.5', '', '-1']` → `(4.5, 2)`
- `texts = []` → `(0, 0)`

In [ ]:
def sum_valid(texts):
    """
    Given strings, return the sum of those that are valid numbers (ints or
    floats) and how many were skipped: (total, skipped).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sum_valid, [
    ((['3', 'x', '2.5', '', '-1'],), (4.5, 2)),
    (([],), (0, 0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A hand-written validity check for every string before converting.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Try `float()` on each and count the `ValueError`s.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** A hand-written validity check for every string before converting.

```python
def sum_valid(texts):
    total, skipped = 0, 0
    for t in texts:
        body = t[1:] if t[:1] in "+-" else t
        if body.replace(".", "", 1).isdigit():
            total += float(t)
        else:
            skipped += 1
    return total, skipped
```

**Cost:** O(n · L) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Try `float()` on each and count the `ValueError`s.

```python
def sum_valid(texts):
    total, skipped = 0, 0
    for t in texts:
        try:
            total += float(t)
        except ValueError:
            skipped += 1
    return total, skipped
```

**Cost:** O(n · L) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A hand-written validity check for every string before converting. That costs O(n · L) time and O(1) space."

"I can do better: Try `float()` on each and count the `ValueError`s. That's O(n · L) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/exceptions/sum-the-valid-numbers)

<a id="q12"></a>
### Q12. Validate an Age

🟡 Interview standard · *raise and except*

Return 'ok' for an age 0–150, otherwise the error message: 'age must be a number', 'age cannot be negative' or 'age is unrealistic'.

**Examples**

- `age = 25` → `'ok'`
- `age = 'x'` → `'age must be a number'`
- `age = -3` → `'age cannot be negative'`

In [ ]:
def check_age(age):
    """
    Return 'ok' for an age 0–150, otherwise the error message: 'age must be a
    number', 'age cannot be negative' or 'age is unrealistic'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(check_age, [
    ((25,), 'ok'),
    (('x',), 'age must be a number'),
    ((-3,), 'age cannot be negative'),
    ((200,), 'age is unrealistic'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Return a message from each if branch.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A checker **raises** ValueError with the message; the caller catches it and reads `str(e)`. This separates detecting a problem from deciding what to do about it.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Return a message from each if branch.

```python
def check_age(age):
    if not isinstance(age, (int, float)):
        return "age must be a number"
    if age < 0:
        return "age cannot be negative"
    if age > 150:
        return "age is unrealistic"
    return "ok"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A checker **raises** ValueError with the message; the caller catches it and reads `str(e)`. This separates detecting a problem from deciding what to do about it.

```python
def check_age(age):
    def validate(a):
        if not isinstance(a, (int, float)):
            raise TypeError("age must be a number")
        if a < 0:
            raise ValueError("age cannot be negative")
        if a > 150:
            raise ValueError("age is unrealistic")

    try:
        validate(age)
    except (TypeError, ValueError) as e:
        return str(e)
    return "ok"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Return a message from each if branch. That costs O(1) time and O(1) space."

"I can do better: A checker **raises** ValueError with the message; the caller catches it and reads `str(e)`. This separates detecting a problem from deciding what to do about it. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/exceptions/validate-an-age)

<a id="q13"></a>
### Q13. Custom Exception: Insufficient Funds

🟡 Interview standard · *Defining exceptions*

Withdraw `amount` from `balance`: return the new balance, or 'insufficient funds' / 'invalid amount' (amount ≤ 0).

**Examples**

- `balance = 100, amount = 30` → `70`
- `balance = 100, amount = 300` → `'insufficient funds'`
- `balance = 100, amount = 0` → `'invalid amount'`

In [ ]:
def withdraw(balance, amount):
    """
    Withdraw amount from balance: return the new balance, or 'insufficient
    funds' / 'invalid amount' (amount ≤ 0).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(withdraw, [
    ((100, 30), 70),
    ((100, 300), 'insufficient funds'),
    ((100, 0), 'invalid amount'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Return special strings straight from the withdraw logic.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Define `class InsufficientFunds(Exception)`; raise it in the logic and catch it at the edge.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Return special strings straight from the withdraw logic.

```python
def withdraw(balance, amount):
    if amount <= 0:
        return "invalid amount"
    if amount > balance:
        return "insufficient funds"
    return balance - amount
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Define `class InsufficientFunds(Exception)`; raise it in the logic and catch it at the edge.

```python
def withdraw(balance, amount):
    class InsufficientFunds(Exception):
        pass

    def take(bal, amt):
        if amt <= 0:
            raise ValueError("invalid amount")
        if amt > bal:
            raise InsufficientFunds("insufficient funds")
        return bal - amt

    try:
        return take(balance, amount)
    except (InsufficientFunds, ValueError) as e:
        return str(e)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Return special strings straight from the withdraw logic. That costs O(1) time and O(1) space."

"I can do better: Define `class InsufficientFunds(Exception)`; raise it in the logic and catch it at the edge. That's O(1) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `balance = 100, amount = 30`

1. InsufficientFunds() returns None
2. take() returns 70
3. withdraw() returns 70

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/exceptions/custom-exception-insufficient-funds)

<a id="q14"></a>
### Q14. Clean-up With finally

🟡 Interview standard · *finally*

Simulate opening a resource, processing `value` (dividing 100 by it) and closing. Return the log, e.g. ['open', 'result 50.0', 'close'] or ['open', 'error', 'close'] — 'close' must always happen.

**Examples**

- `value = 2` → `['open', 'result 50.0', 'close']`
- `value = 0` → `['open', 'error', 'close']`

In [ ]:
def with_cleanup(value):
    """
    Simulate opening a resource, processing value (dividing 100 by it) and
    closing. Return the log, e.g. ['open', 'result 50.0', 'close'] or ['open',
    'error', 'close'] — 'close' must always happen.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(with_cleanup, [
    ((2,), ['open', 'result 50.0', 'close']),
    ((0,), ['open', 'error', 'close']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Write 'close' in every branch by hand (easy to forget one).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`finally` runs no matter what — success, error or return.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Write 'close' in every branch by hand (easy to forget one).

```python
def with_cleanup(value):
    log = ["open"]
    if value == 0:
        log.append("error")
        log.append("close")
        return log
    log.append(f"result {100 / value}")
    log.append("close")
    return log
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `finally` runs no matter what — success, error or return.

```python
def with_cleanup(value):
    log = ["open"]
    try:
        log.append(f"result {100 / value}")
    except ZeroDivisionError:
        log.append("error")
    finally:
        log.append("close")
    return log
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Write 'close' in every branch by hand (easy to forget one). That costs O(1) time and O(1) space."

"I can do better: `finally` runs no matter what — success, error or return. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/exceptions/clean-up-with-finally)

<a id="q15"></a>
### Q15. First Valid Attempt

🟡 Interview standard · *Retry loop*

A user typed several attempts. Return `(number, attempts_used)` for the first valid integer, or `(None, len(attempts))`.

**Examples**

- `attempts = ['abc', '', '7', '9']` → `(7, 3)`
- `attempts = ['x']` → `(None, 1)`

In [ ]:
def first_valid(attempts):
    """
    A user typed several attempts. Return (number, attempts_used) for the first
    valid integer, or (None, len(attempts)).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(first_valid, [
    ((['abc', '', '7', '9'],), (7, 3)),
    ((['x'],), (None, 1)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Validate each attempt by hand with string checks.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Loop over the attempts with `enumerate`; return from inside `try` on success.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Validate each attempt by hand with string checks.

```python
def first_valid(attempts):
    count = 0
    for a in attempts:
        count += 1
        s = a.strip()
        if s and (s.isdigit() or (s[0] == "-" and s[1:].isdigit())):
            return int(s), count
    return None, count
```

**Cost:** O(n · L) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Loop over the attempts with `enumerate`; return from inside `try` on success.

```python
def first_valid(attempts):
    for used, a in enumerate(attempts, start=1):
        try:
            return int(a), used
        except ValueError:
            continue
    return None, len(attempts)
```

**Cost:** O(n · L) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Validate each attempt by hand with string checks. That costs O(n · L) time and O(1) space."

"I can do better: Loop over the attempts with `enumerate`; return from inside `try` on success. That's O(n · L) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `attempts = ['abc', '', '7']`

1. ValueError raised: invalid literal for int() with base 10: 'abc'
2. ValueError raised: invalid literal for int() with base 10: ''
3. first\_valid() returns (7, 3)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/exceptions/first-valid-attempt)

<a id="q16"></a>
### Q16. Several Kinds of Error

🟡 Interview standard · *Multiple except blocks*

Return `int(items[i])`, or 'missing' if i is out of range, or 'not a number' if the item can't be converted.

**Examples**

- `items = ['5', 'x'], i = 0` → `5`
- `items = ['5', 'x'], i = 1` → `'not a number'`
- `items = ['5'], i = 4` → `'missing'`

In [ ]:
def item_as_int(items, i):
    """
    Return int(items[i]), or 'missing' if i is out of range, or 'not a number'
    if the item can't be converted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(item_as_int, [
    ((['5', 'x'], 0), 5),
    ((['5', 'x'], 1), 'not a number'),
    ((['5'], 4), 'missing'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check the index, then check the text, each with its own if.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One try with two `except` blocks, one per error type.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check the index, then check the text, each with its own if.

```python
def item_as_int(items, i):
    if i >= len(items) or i < -len(items):
        return "missing"
    text = items[i].strip()
    if not text.lstrip("-").isdigit():
        return "not a number"
    return int(text)
```

**Cost:** O(L) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** One try with two `except` blocks, one per error type.

```python
def item_as_int(items, i):
    try:
        return int(items[i])
    except IndexError:
        return "missing"
    except ValueError:
        return "not a number"
```

**Cost:** O(L) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check the index, then check the text, each with its own if. That costs O(L) time and O(1) space."

"I can do better: One try with two `except` blocks, one per error type. That's O(L) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/exceptions/several-kinds-of-error)

<a id="q17"></a>
### Q17. Totals From CSV

🟡 Interview standard · *csv module*

The file has lines 'name,amount'. Return the total amount per name, skipping lines whose amount isn't a number.

**Examples**

- `text = 'ana,10\nben,5\nana,2.5\ncy,oops\n'` → `{'ana': 12.5, 'ben': 5.0}`
- `text = ''` → `{}`

In [ ]:
def csv_totals(text):
    """
    The file has lines 'name,amount'. Return the total amount per name, skipping
    lines whose amount isn't a number.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(csv_totals, [
    (('ana,10\nben,5\nana,2.5\ncy,oops\n',), {'ana': 12.5, 'ben': 5.0}),
    (('',), {}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Split each line on ',' by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`csv.reader` handles quoting and commas; `try/except` skips bad amounts.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Split each line on ',' by hand.

```python
def csv_totals(text):
    totals = {}
    for line in text.split("\n"):
        if line == "":
            continue
        parts = line.split(",")
        name, amount = parts[0], parts[1]
        if amount.replace(".", "", 1).isdigit():
            totals[name] = totals.get(name, 0) + float(amount)
    return totals
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `csv.reader` handles quoting and commas; `try/except` skips bad amounts.

```python
def csv_totals(text):
    import io
    import csv
    totals = {}
    with io.StringIO(text) as f:
        for name, amount in csv.reader(f):
            try:
                totals[name] = totals.get(name, 0) + float(amount)
            except ValueError:
                continue
    return totals
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Split each line on ',' by hand. That costs O(n) time and O(k) space."

"I can do better: `csv.reader` handles quoting and commas; `try/except` skips bad amounts. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/exceptions/totals-from-csv)

<a id="q18"></a>
### Q18. Read key=value Settings

🟡 Interview standard · *Unpacking errors*

Parse lines like 'port=8080' into a dict, skipping blank lines, comments (#) and malformed lines.

**Examples**

- `text = 'port=8080\n# note\nbad line\nhost=local\n\n'` → `{'port': '8080', 'host': 'local'}`
- `text = ''` → `{}`

In [ ]:
def read_settings(text):
    """
    Parse lines like 'port=8080' into a dict, skipping blank lines, comments (#)
    and malformed lines.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(read_settings, [
    (('port=8080\n# note\nbad line\nhost=local\n\n',), {'port': '8080', 'host': 'local'}),
    (('',), {}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check for '=' and comments with ifs, then split.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`partition('=')` splits once; skip lines where the separator is missing.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check for '=' and comments with ifs, then split.

```python
def read_settings(text):
    result = {}
    for line in text.split("\n"):
        line = line.strip()
        if line == "" or line[0] == "#":
            continue
        if "=" not in line:
            continue
        index = line.index("=")
        result[line[:index].strip()] = line[index + 1:].strip()
    return result
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `partition('=')` splits once; skip lines where the separator is missing.

```python
def read_settings(text):
    result = {}
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        key, sep, value = line.partition("=")
        if sep:
            result[key.strip()] = value.strip()
    return result
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check for '=' and comments with ifs, then split. That costs O(n) time and O(k) space."

"I can do better: `partition('=')` splits once; skip lines where the separator is missing. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/exceptions/read-key-value-settings)

<a id="q19"></a>
### Q19. Parse JSON Safely

🟡 Interview standard · *json + JSONDecodeError*

Parse a JSON text and return its 'name' field, or 'invalid json' / 'no name'.

**Examples**

- `text = '{"name": "Ana", "age": 21}'` → `'Ana'`
- `text = '{"age": 21}'` → `'no name'`
- `text = '{oops'` → `'invalid json'`

In [ ]:
def json_name(text):
    """
    Parse a JSON text and return its 'name' field, or 'invalid json' / 'no
    name'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(json_name, [
    (('{"name": "Ana", "age": 21}',), 'Ana'),
    (('{"age": 21}',), 'no name'),
    (('{oops',), 'invalid json'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check that the text looks like an object, then parse it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Let `json.loads` decide and catch `json.JSONDecodeError`; `dict.get` handles the missing key.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check that the text looks like an object, then parse it.

```python
def json_name(text):
    import json
    t = text.strip()
    if not (t.startswith("{") and t.endswith("}")):
        return "invalid json"
    data = json.loads(t)
    if "name" in data:
        return data["name"]
    return "no name"
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Let `json.loads` decide and catch `json.JSONDecodeError`; `dict.get` handles the missing key.

```python
def json_name(text):
    import json
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        return "invalid json"
    return data.get("name", "no name")
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check that the text looks like an object, then parse it. That costs O(n) time and O(n) space."

"I can do better: Let `json.loads` decide and catch `json.JSONDecodeError`; `dict.get` handles the missing key. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/exceptions/parse-json-safely)

<a id="q20"></a>
### Q20. Make Your Own Context Manager

🔴 Stretch · *\_\_enter\_\_ / \_\_exit\_\_*

Return the log of entering, doing the work and leaving a 'session', where leaving must happen even if the work fails: e.g. ['enter', 'work ok', 'exit'] or ['enter', 'exit', 'handled'].

**Examples**

- `fail = False` → `['enter', 'work ok', 'exit']`
- `fail = True` → `['enter', 'exit', 'handled']`

In [ ]:
def session_log(fail):
    """
    Return the log of entering, doing the work and leaving a 'session', where
    leaving must happen even if the work fails: e.g. ['enter', 'work ok',
    'exit'] or ['enter', 'exit', 'handled'].
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(session_log, [
    ((False,), ['enter', 'work ok', 'exit']),
    ((True,), ['enter', 'exit', 'handled']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

try/finally written out by hand around the work.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`contextlib.contextmanager` turns a generator into a `with`-able object; the code after `yield` always runs.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** try/finally written out by hand around the work.

```python
def session_log(fail):
    log = ["enter"]
    try:
        try:
            if fail:
                raise RuntimeError("boom")
            log.append("work ok")
        finally:
            log.append("exit")
    except RuntimeError:
        log.append("handled")
    return log
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `contextlib.contextmanager` turns a generator into a `with`-able object; the code after `yield` always runs.

```python
def session_log(fail):
    from contextlib import contextmanager
    log = []

    @contextmanager
    def session():
        log.append("enter")
        try:
            yield
        finally:
            log.append("exit")

    try:
        with session():
            if fail:
                raise RuntimeError("boom")
            log.append("work ok")
    except RuntimeError:
        log.append("handled")
    return log
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: try/finally written out by hand around the work. That costs O(1) time and O(1) space."

"I can do better: `contextlib.contextmanager` turns a generator into a `with`-able object; the code after `yield` always runs. That's O(1) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `fail = True`

1. session() returns None
2. RuntimeError raised: boom
3. RuntimeError raised: boom
4. session() returns None
5. session\_log() returns ['enter', 'exit', 'handled']

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/exceptions/make-your-own-context-manager)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Error vs exception?</b></summary>

'Error' is the general term for a problem. Syntax errors stop the program before it runs. Exceptions are run-time error objects that can be caught and handled with try/except.

</details>

<details>
<summary><b>2. What is the difference between except and finally?</b></summary>

`except` runs only when a matching exception occurs. `finally` always runs, whether or not an exception occurred, and is used for clean-up.

</details>

<details>
<summary><b>3. When does the else block of a try run?</b></summary>

Only when the try block completes without raising an exception. It keeps success code out of the try block so its errors aren't caught by mistake.

</details>

<details>
<summary><b>4. Why is a bare except: bad?</b></summary>

It catches everything, including KeyboardInterrupt, SystemExit and bugs like NameError, which hides real problems. Catch specific exceptions, or at least `except Exception`.

</details>

<details>
<summary><b>5. How do you create a custom exception?</b></summary>

Subclass Exception: `class MyError(Exception): pass`, then `raise MyError('message')`.

</details>

<details>
<summary><b>6. Why use with open(...)?</b></summary>

The with statement closes the file automatically when the block ends, even if an exception occurs, preventing leaked file handles and unflushed data.

</details>

<details>
<summary><b>7. What does raise ... from e do?</b></summary>

It raises a new exception while recording the original as its cause (`__cause__`), so the traceback shows both.

</details>

<details>
<summary><b>8. raise vs assert?</b></summary>

`raise` signals an error condition you expect could happen (bad input). `assert` checks something that should always be true if the code is correct; asserts can be turned off with -O.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Errors are normal
- [ ] I can explain: try and except
- [ ] I can explain: else, finally and several excepts
- [ ] I can explain: Raising your own errors
- [ ] I can explain: Reading and writing files
- [ ] I can explain: Context managers and structured data files

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Several Kinds of Error
- [ ] [Q17](#q17) Totals From CSV
- [ ] [Q18](#q18) Read key=value Settings
- [ ] [Q19](#q19) Parse JSON Safely
- [ ] [Q20](#q20) Make Your Own Context Manager

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.